# Alert 생성기 실험 ⓪ — 서비스 이진 모델로 점수 내기

01 ~ 03번 실험이 읽을 **거래별 점수 파일**을 서비스 이진 모델(`multi_fraudgt_hgb`)로 만든다.
모델 코드와 가중치는 서비스 저장소(`money_laundry/backend/inference-server`)의 것을 그대로 불러 쓴다. 새로 학습하지 않는다.

**서비스와 같게 맞춘 것**
- 하루씩 채점한다. 그날 거래의 피처는 **처음부터 그날 마감까지의 거래 전체**로 만든다. 그날 뒤의 거래는 보지 않는다
- 배치 크기와 샘플링 시드(서비스 `spec.py` 값), 계좌 정보 규칙(소유주 유형 6종, 국가)

**서비스와 다른 것**
- 계좌 번호(`node_id`)를 매기는 순서가 서비스 DB와 다르다. 이웃 샘플링이 이 번호를 따라가서, 같은 거래라도 점수가 서비스와 소수점 수준에서 다를 수 있다

**평가 구간**
- 데이터 첫날부터 `WARMUP_DAYS` 일은 점수를 내지 않는다. 계좌·상대 이력이 쌓이기 전이라 피처가 비어 있어 오탐이 많다(2026-11-23 결정)
- 그 기간의 거래도 **피처를 만들 때는 전부 본다**. 점수(씨앗)만 안 낼 뿐이다

먼저 `code/01_prepare_parquet.ipynb` 를 이 데이터셋으로 돌려 `parquets/<DATASET>/` 을 만들어 둬야 한다.

## 0. 설정

In [ ]:
DATASET = 'HI-Medium'
WARMUP_DAYS = 7                  # 데이터 첫날부터 이 일수는 점수를 내지 않는다
EVAL_END = None                  # 평가 구간 마지막 날 'YYYY-MM-DD'. None 이면 3절의 제안값을 쓴다

DEVICE = 'cuda'                  # 'cpu' / 'cuda' / 'cuda:2'. 서비스 기본은 cpu 다
THREADS, MEMORY = 8, '24GB'      # 전처리(DuckDB) 스레드 수와 메모리 한도. 서비스 기본은 2 / 2GB
MAX_DAYS = 1                     # 5절에서 이번 실행에 채점할 날 수. 처음에는 1 로 하루만 돌려 시간을 본다. None 이면 남은 날 전부

In [ ]:
from pathlib import Path
import json, time
import duckdb, numpy as np, pandas as pd
import svc_lib as SL

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'parquets').is_dir()), None)
assert ROOT is not None, 'parquets/ 폴더를 못 찾음. 데이터가 있는 GPU 서버의 저장소 안에서 열어야 한다'
BASE = ROOT / 'parquets' / DATASET
for name in ('transactions.parquet', 'kyc.parquet', 'patterns.parquet'):
    assert (BASE / name).is_file(), f'{BASE / name} 이 없다. code/01_prepare_parquet.ipynb 를 DATASET = {DATASET!r} 로 먼저 돌린다'
SCORE_DIR = BASE / 'service_scores'              # 점수 파일. 크기가 커서 parquets/ 아래에 둔다(커밋되지 않는다)
DAY_DIR, WORK = SCORE_DIR / 'days', SCORE_DIR / 'work'
OUT = Path.cwd() / 'results' / DATASET / 'service_scores'      # 작은 요약 표만 (커밋된다)
for folder in (DAY_DIR, WORK, OUT):
    folder.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
con.execute(f"SET threads={THREADS}"); con.execute(f"SET memory_limit='{MEMORY}'")
print(BASE)

## 1. 서비스 모델 불러오기
`money_laundry` 저장소가 이 저장소와 나란히 있어야 한다. 서비스가 고정한 라이브러리 버전과 지금 환경을 같이 본다.
`같음` 이 False 인 줄이 있으면 2절의 자기 점검 결과를 꼭 확인한다. 특히 scikit-learn 이 다르면 트리 모델이 다르게 읽힐 수 있다.

In [ ]:
SERVER = SL.find_server()
print(SERVER)
print(SL.git_commit(SERVER))
SL.versions(SERVER)

In [ ]:
t = time.time()
M = SL.load_model(SERVER, DEVICE, THREADS)       # 가중치 체크섬도 여기서 확인된다
S = SL.worker_settings(SERVER)                   # 의심 기준, 씨앗 보강 값 (백엔드 spec.py)
MODEL_VERSION, THRESHOLD = M.spec.MODEL_VERSION, S['threshold']
print(f'{MODEL_VERSION} | 장치 {DEVICE} | 배치 {M.spec.BATCH_SIZE} | 시드 {M.spec.SAMPLING_SEED} | 의심 기준 {THRESHOLD} | 보강 {S["seed_low"]} / {S["chain_floor"]}  ({time.time()-t:.0f}s)')

## 2. 자기 점검
패키지에 딸린 예제(거래 81건)를 5절과 같은 함수로 채점해서 패키지가 적어 둔 점수와 비교한다.
`최대_차이` 가 2e-6 안이면 서비스와 같은 점수가 나오는 환경이다. GPU 에서는 조금 더 벌어질 수 있는데, 1e-4 를 넘으면 환경(라이브러리 버전)을 의심한다.

In [ ]:
check = SL.self_check(M, WORK, THREADS, MEMORY)
check

## 3. 날짜별 거래 수 → 평가 구간
- 시작 = 데이터 첫날 + `WARMUP_DAYS`
- 끝 = 거래 수가 (가장 많은 날의 10%) 이상인 마지막 날. 그 뒤는 정상 거래가 끊기고 세탁 거래만 이어지는 꼬리라 평가에 쓰지 않는다

표를 보고 끝 날짜가 이상하면 0절의 `EVAL_END` 에 직접 적는다.

In [ ]:
days = SL.daily_counts(con, BASE / 'transactions.parquet')
start, end = SL.propose_window(days, WARMUP_DAYS)
EVAL_START, EVAL_END = start, (EVAL_END or end)
day = pd.to_datetime(days.day)
days['구분'] = np.select([day < EVAL_START, day <= EVAL_END], ['이력만 (점수 안 냄)', '평가'], '꼬리 (안 씀)')
print(f'평가 구간 {EVAL_START} ~ {EVAL_END}')
days

In [ ]:
DAYS = [str(d.date()) for d in pd.date_range(EVAL_START, EVAL_END)]
part = days.groupby('구분')[['거래', '세탁']].sum()
print(f'평가 {len(DAYS)}일')
part

## 4. 모델 입력 파일
서비스의 `materials.normalize` 와 같은 규칙으로 두 파일을 쓴다.
- 거래: 모델이 쓰는 열만, 평가 구간 끝까지만. **정답 열은 넣지 않는다**
- 계좌: 소유주 유형이 학습한 6종이 아니면 비운다

In [ ]:
TX_MODEL, KYC_MODEL, info = SL.write_inputs(con, BASE / 'transactions.parquet', BASE / 'kyc.parquet', SCORE_DIR, M, EVAL_END)
print(M.entity_types)
info

## 5. 하루씩 채점
하루가 서비스의 하루 실행 한 번이다: 처음부터 그날까지의 거래로 피처를 다시 만들고(전처리), 그날 거래 전부의 점수를 낸다(추론).

- 끝난 날은 `days/<날짜>.parquet` 로 남고, 다시 실행하면 건너뛴다. 중간에 끊겨도 이어 돌릴 수 있다
- **처음에는 `MAX_DAYS = 1` 로 하루만 돌려 시간과 메모리를 본다.** 뒤 날짜일수록 이력이 길어 전처리가 오래 걸린다
- 괜찮으면 0절에서 `MAX_DAYS = None` 으로 바꾸고 이 셀만 다시 돌린다 (커널을 다시 시작했다면 0 ~ 4절부터)

In [ ]:
todo = [d for d in DAYS if not (DAY_DIR / f'{d}.parquet').exists()]
print(f'전체 {len(DAYS)}일 | 끝난 날 {len(DAYS) - len(todo)} | 이번에 돌릴 날 {len(todo[:MAX_DAYS])}')
log = []
for d in todo[:MAX_DAYS]:
    scores, took = SL.score_day(M, TX_MODEL, KYC_MODEL, d, WORK, THREADS, MEMORY)
    tmp = DAY_DIR / f'{d}.tmp'
    scores.to_parquet(tmp, index=False); tmp.rename(DAY_DIR / f'{d}.parquet')      # 다 쓴 뒤에 이름을 바꾼다
    log.append(dict(날짜=d, 거래=len(scores), **took, 기준_이상=int((scores.score >= THRESHOLD).sum())))
    print(log[-1])
pd.DataFrame(log)

## 6. 합치고 정답 붙이기
평가 구간의 모든 날이 끝나 있어야 한다. 거래 파일의 정답(`is_laundering`)을 `y` 로 붙인다.

In [ ]:
missing = [d for d in DAYS if not (DAY_DIR / f'{d}.parquet').exists()]
assert not missing, f'아직 채점하지 않은 날이 {len(missing)}일 있다: {missing[:5]} ... 5절을 MAX_DAYS = None 으로 다시 돌린다'
scored = pd.concat([pd.read_parquet(DAY_DIR / f'{d}.parquet') for d in DAYS], ignore_index=True)
truth = con.execute(f"""SELECT edge_id, CAST(is_laundering AS INTEGER) AS y, CAST(ts AS DATE) AS day
                        FROM read_parquet({SL.q(BASE / 'transactions.parquet')})
                        WHERE CAST(ts AS DATE) BETWEEN DATE {SL.q(EVAL_START)} AND DATE {SL.q(EVAL_END)}""").df()
table = truth.merge(scored, on='edge_id', how='left').sort_values('edge_id').reset_index(drop=True)
print(f'평가 구간 거래 {len(truth):,} | 점수 {len(scored):,} | 점수 없는 거래 {int(table.score.isna().sum()):,} (0 이어야 정상) | 세탁 {int(table.y.sum()):,}')
assert len(scored) == len(truth) and table.score.notna().all() and scored.edge_id.is_unique

## 7. 빠른 확인
자세한 분석은 01 ~ 03번에서 한다. 여기서는 점수가 말이 되는지만 본다.
- `결합` 이 서비스가 쓰는 최종 점수다. `그래프 모델만`, `트리만` 은 결합 전 두 모델의 점수다(기준값은 결합 점수용이라 AP 만 본다)

In [ ]:
from sklearn.metrics import average_precision_score
y = table.y.to_numpy()
ap = pd.DataFrame([dict(점수=name, AP=round(average_precision_score(y, table[col]), 4))
                   for name, col in [('결합', 'score'), ('그래프 모델만', 'gnn_score'), ('트리만', 'hgb_score')]])
summary = pd.DataFrame([SL.at_threshold('평가 구간 전체', y, table.score, THRESHOLD)])
display(ap)
summary

In [ ]:
# 날짜별. 앞 날짜의 정밀도가 뒤보다 많이 낮으면 WARMUP_DAYS 를 늘릴지 본다
by_day = pd.DataFrame([SL.at_threshold(str(d)[:10], g.y, g.score, THRESHOLD) for d, g in table.groupby('day')])
by_day

In [ ]:
# 패턴별 재현율 (의심 기준에서). NO_PATTERN = 패턴 파일에 없는 세탁 거래
pattern = con.execute(f"SELECT edge_id, any_value(pattern_type) AS pattern_type FROM read_parquet({SL.q(BASE / 'patterns.parquet')}) GROUP BY 1").df()
laundering = table[table.y == 1].merge(pattern, on='edge_id', how='left').fillna({'pattern_type': 'NO_PATTERN'})
by_pattern = (laundering.assign(잡음=laundering.score >= THRESHOLD).groupby('pattern_type')
              .agg(세탁_거래=('잡음', 'size'), 잡은_거래=('잡음', 'sum')).assign(재현율=lambda t: (t.잡은_거래 / t.세탁_거래).round(4)).reset_index())
by_pattern

## 8. 저장
- 점수 파일: `parquets/<DATASET>/service_scores/<모델 버전>_test_scores.parquet` (`edge_id, y, score, gnn_score, hgb_score`). 01 ~ 03번이 읽는 모양이다
- 설정 파일: 같은 폴더의 `<모델 버전>_results.json`. 의심 기준이 `['val']['threshold']` 에 있어 21번 결과와 같은 줄로 읽힌다
- 요약 표: `results/<DATASET>/service_scores/`

In [ ]:
scores_path = SCORE_DIR / f'{MODEL_VERSION}_test_scores.parquet'
table[['edge_id', 'y', 'score', 'gnn_score', 'hgb_score']].to_parquet(scores_path, index=False)
record = dict(name=MODEL_VERSION, dataset=DATASET, val=dict(threshold=THRESHOLD), seed_low=S['seed_low'], chain_floor=S['chain_floor'],
              eval_window=[EVAL_START, EVAL_END], warmup_days=WARMUP_DAYS, data_start=str(days.day.min())[:10],
              batch_size=M.spec.BATCH_SIZE, sampling_seed=M.spec.SAMPLING_SEED, device=DEVICE,
              service_commit=SL.git_commit(SERVER), self_check=check, inputs=info, rows=len(table), laundering=int(table.y.sum()))
json.dump(record, open(SCORE_DIR / f'{MODEL_VERSION}_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
json.dump(record, open(OUT / 'run.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
days.to_csv(OUT / 'days.csv', index=False); ap.to_csv(OUT / 'ap.csv', index=False); summary.to_csv(OUT / 'summary.csv', index=False)
by_day.to_csv(OUT / 'by_day.csv', index=False); by_pattern.to_csv(OUT / 'by_pattern.csv', index=False)
print(scores_path, f'{scores_path.stat().st_size / 1e6:.0f} MB')
print(sorted(p.name for p in OUT.iterdir()))